# 서울 특별시 공공자전거 이용정보 데이터 분석
- 사용 데이터: 서울특별시 공공자전거 이용정보(월별)
- 데이터 측정 기간 : 2024년 7월 - 12월

## 1. 데이터 불러오기

In [ ]:
import pandas as pd
data = pd.read_csv("data/서울특별시 공공자전거 이용정보(월별)_24.7-12.csv", encoding="cp949")
original_data = data.copy()
data.head()

## 2. 데이터 정보 확인 및 전처리

In [ ]:
data.info()

### 1) 데이터 형 변환 하기
- 연산이 필요 없는 데이터 변환하기
- 카테고리(범주형 데이터)를 확인하고, 데이터를 일정하게 맞추기

대여소 번호는 연산이 필요 없다.
즉, 통계 계산 데이터를 얻을 필요가 없는 고유값 데이터이다.

`describe` 함수 사용시, 
대여소 번호에 대한 통계 데이터를 object로 취급할 수 있도록 형 변환을 하자

#### 데이터 형 변환을 해야 하는 이유

int, float 타입으로 되어 있는 열은 `.describe()`라는 함수를 이용할 때 통계에 이용된다. 하지만 대여소번호와 같이 고유의 이름을 의미하는 경우에는 통계가 필요없기 때문에 object로 형 변환을 해줄 필요가 있다.

In [ ]:
# data["대여소번호"] = data["대여소번호"].apply("object")
data["대여소번호"] = data["대여소번호"].astype("object")
data.info()

대여일자를 datetime으로 변경하려고 했지만
대여일자를 확인해보니 연-월 정보만 가지고 있다.

```python
# 202407
data["대여일자"] = pd.to_datetime(data["대여일자"], format="%Y%m")
data["대여일자"]
```

하지만, 해당 데이터는 월별 데이터이므로
대여일자도 범주형 데이터라고 할 수 있다

따라서, object 로 변환하자

In [ ]:
data["대여일자"] = data["대여일자"].astype("object")

In [ ]:
# astype 한 번에 하기
data = data.astype({"대여일자" : object, "대여소번호" : object})

### 2) 카테고리 변수 unique 확인

* 카테고리로 표현되는 변수는 성별, 연령대코드이다.
* 어떤 종류가 있는지 살펴보려고 한다.

In [ ]:
data.isna()

In [ ]:
data["성별"].unique()

- 성별에 결측치 데이터가 많다. 이는 응답하지 않은 케이스로 해당 데이터를 Other을 의미하는 "O" 로 만들자
- 성별 데이터에 대무자와 소문자가 섞여있어 하나로 통일시키자 (대문자로 통일하겠다)

In [ ]:
data["성별"] = data["성별"].str.upper()
data["성별"] = data["성별"].fillna("O")
data["성별"].unique()

성별에 대한 결측치 데이터가 사라졌음을 확인할 수 있다.

In [ ]:
data["성별"] = data["성별"].astype("category")
data["성별"].unique()

In [ ]:
data["연령대코드"].unique()
data["연령대코드"] = data["연령대코드"].astype("category")

## 3) 결측치 확인하기

In [ ]:
nan_result = data.isna().sum(axis=0)
nan_result

- 성별 데이터의 결측치가 195384건 : 약 30%

In [ ]:
nan_result["성별"]

### 최종적으로 데이터 확인 다시 하기

In [ ]:
original_data.info()

In [ ]:
data.info()

In [ ]:
data.describe()

### 참고

In [ ]:
# 숫자형 데이터만 뽑았을 때, 총 결측치 수
import numpy as np

numeric_df = data.select_dtypes(include=[np.number])
nan_count = np.isnan(numeric_df).sum().sum()
print(f"숫자형에서 NaN 개수: {nan_count}")

## 4. 데이터 탐색

In [ ]:
data.head()

### 1) 새로운 열 만들기

<div>
    <img src="https://www.dropbox.com/scl/fi/3s8gvus21mqhcbqkz38zf/.png?rlkey=1kw3p6nleovg9dw6m82ik9izo&dl=1" width="80%">
</div>

In [ ]:
# 건당 이동거리(M), 건당 이용시간(분)

data["건당 이동거리(M)"] = data["이동거리(M)"] / data["이용건수"]
data["건당 이용시간(분)"] = data["이용시간(분)"] / data["이용건수"]
data.head()

### 2) 데이터 요약하기

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
# head(6): 2024년 7월에 102번 대여소, 일일권, 성별 없는 건들의 집계 데이터
test_data = data.head(6)
test_data

In [ ]:
# 평균 이동거리(M)
# 이동거리의 합 / 이용건 수
test_data["이동거리(M)"].sum() / test_data["이용건수"].sum()

In [ ]:
# 건당 이동거리(M)
# 각 행의 이동거리(M) / 이용건수
# 이 의미는 무엇일까?
test_data["건당 이동거리(M)"].mean()

### 3) 데이터 시각화

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import koreanize_matplotlib

### 이전 코드 복습해보기 
- 목표: 대여일자에 따른 행 카운트 세기
- 판다스: .value_counts()를 통해 대여일자의 빈도표 출력해보기
- 시각화: sns.countplot을 이용하여 대여일자에 대한 막대그래프 그려보기

In [ ]:
count_by_month = data["대여일자"].value_counts()
count_by_month

In [ ]:
plt.figure(figsize=(3,3))
count_by_month.plot(kind="bar")
plt.title("이전 코드 복습해보기")
plt.show()

- 이 행은

In [ ]:
plt.figure(figsize=(7,3))
sns.countplot(data=data, x="대여일자")
plt.title("이전 코드 복습해보기")
plt.show()

#### 이전코드 복습을 다른 방법으로 하기

In [ ]:
cm = sns.color_palette('hls', len(data['대여일자'].unique()))
ax = sns.barplot(data=count_by_month, palette=cm, legend=False)

ax.set_ylim(90000,110000)
ax.set_yticks(range(90000, 110000, 2000))
plt.title("이전 코드 복습해보기")
plt.show()

In [ ]:
month_group = data.groupby("대여일자", observed=False)
num_by_month = month_group["이용건수"].sum()
num_by_month

In [ ]:
# cm = sns.color_palette('hls', len(data['대여일자'].unique()))
ax = sns.barplot(data=num_by_month)

ax.set_ylim(2000000,4700000)
ax.set_yticks(range(2000000, 4700000, 200000))
ax.set_title("월별 이용 건수")
plt.show()

### 성별이 남자인 사람의 평균 이동거리, 평균 이용시간
- 목표: 데이터 요약 
- 성별이 남자인 데이터 출력하기 data_m
- data_m에서 평균 이동거리, 평균 이용시간 추출한 후 .mean() 사용해보기

#### ⭐ 고민 해볼 것
- 이동거리, 이용시간은 어떻게 집계된 데이터일까?
    - 이동거리 = 대여를 시작한 장소로부터, 대여를 끝낸 장소까지의 위치 데이터를 직선거리로 계산 (실시간 gps로 더 정확한 거리를 계산할 수 도 있을 듯)
    - 이용시간 = 대여를 시작한 시간부터, 대여를 끝낸 시간까지의 데이터 간의 차이로 계산
    - 두 데이터 모두 각 집단의 이동거리의 합산
- 건당 이동거리, 건당 이용시간이 왜 필요할까?
    - 각 집단의 대표값 설정을 위해서?

#### 정확한 계산식

In [ ]:
# 평균 이동거리(M)
# 이동거리의 합 / 이동거리 건수
data["이동거리(M)"].sum() / data["이용건수"].sum()

#### 평균의 평균값으로 계산

평균의 평균값으로 계산할 경우 극단값이 존재하면, 
신뢰할 수 있는 값이 나오지 않을 수도 있으므로 주의해야한다.

In [ ]:
data_m = data[data["성별"] == "M"]
data_m[["건당 이동거리(M)", "건당 이용시간(분)"]].mean()

In [ ]:
data_m["이용시간(분)"].sum() / data_m["이용건수"].sum()

### 성별로 평균 이동거리, 평균 이용시간

In [ ]:
data.groupby("성별", observed=False)[["이동거리(M)", "이용시간(분)"]].mean()

In [ ]:
data.groupby("성별", observed=False)["이동거리(M)"].agg(func=["min", "mean", "max"])

#### 정확한 계산식

In [ ]:
data.groupby("성별")["이동거리(M)"].sum()

#### 평균의 평균값으로 계산

In [ ]:
data.groupby("성별")["건당 이동거리(M)"].mean()

### 연령대별 이용건수

In [ ]:
age_group = data.groupby("연령대코드", observed=False)
sum_age_data = age_group["이용건수"].sum()
sum_age_data

In [ ]:
sum_age_data.plot(kind="bar")

In [ ]:
data.groupby("연령대코드")["이용건수"].agg(func=["sum"]).sort_index()

In [ ]:
cm = sns.color_palette('hls', len(data['대여일자'].unique()))
ax = sns.barplot(data=sum_age_data, order=["~10대", "20대", "30대", "40대", "50대", "60대", "70대이상", "기타"], palette=cm)

ax.set_ylim(0,7000000)
ax.set_yticks(range(0, 7000000, 500000))
ax.set_title("연령대별 이용건수")
plt.show()

### 대여소별 이용건수

In [ ]:
# 대여소 별 이용건수
loc_group = data.groupby("대여소번호", observed=False)
num_by_loc = loc_group["이용건수"].sum()

loc_id = data[["대여소번호", "대여소명"]].drop_duplicates()

pd.merge(num_by_loc, loc_id, on="대여소번호")

### 연령대별 이동거리, 이용시간의 평균

In [ ]:
data.groupby("연령대코드")[["이동거리(M)", "이용시간(분)"]].mean()

In [ ]:
# 월별 어느 연령대가 가장 많이 이용했는가
# 각 대여소별 평균 이동 거리를 분석하여 용도 추론
# 대여소별 이용자 연령대
# 대여소별 이용건수
# 연령대별 이용건수가 많은 상위 대여소 리스트
# 대여소별 건당 이용시간 평균
# 수거 용이 시간 분석

# Hint: .groupby(), .pivot_table(), .crosstab()

### 상관관계 분석

In [ ]:
import pandas as pd
import numpy as np

data.select_dtypes(include=np.number).corr()

### 4) 지도 시각화

`uv add folium`

In [ ]:
data.head()

In [ ]:
# 공공자전거 대여소 정보(api수집).csv
sub_data = pd.read_csv("data/공공자전거 대여소 정보(api수집).csv")
sub_data.head()

In [ ]:
# RENT_NO, RENT_ID_NM, STA_LAT, STA_LONG 열만 가지고 오기
sub_data = sub_data[["RENT_NO", "RENT_ID_NM", "STA_LAT", "STA_LONG"]]
sub_data.head()

In [ ]:
sub_data.rename(columns={
    "RENT_NO" : "대여소번호",
    "RENT_ID_NM" : "대여소명",
    "STA_LAT" : "위도", 
    "STA_LONG" : "경도"
}, inplace=True)
sub_data.head()

In [ ]:
# new_data에서 대여소별로 이동거리(M), 이용시간(분), 이용건수의 합 구하기

summary_data = data.groupby("대여소명")[["이동거리(M)", "이용시간(분)", "이용건수"]].sum()
summary_data

In [ ]:
result = pd.pivot_table(data,                # 피벗할 데이터프레임
                index = '대여소명',    # 행 위치에 들어갈 열
                # columns = ["이동거리(M)", "이용시간(분)", "이용건수"],    # 열 위치에 들어갈 열
                values = ["이동거리(M)", "이용시간(분)", "이용건수"],     # 데이터로 사용할 열
                aggfunc = ['mean', 'sum'])   # 데이터 집계함수
result.head()

In [ ]:
result = pd.pivot_table(
                data = data,                # 피벗할 데이터프레임
                index = '대여소명',    # 행 위치에 들어갈 열
                columns = "성별",    # 열 위치에 들어갈 열
                values = "이용건수",     # 데이터로 사용할 열
                aggfunc = ['sum'],
                observed=False)   # 데이터 집계함수
result.head()

In [ ]:
import pandas as pd

result = pd.crosstab(
    index = data["대여소명"], 
    columns = data["연령대코드"], 
    values=None, 
    aggfunc=None, 
    margins=True, 
    margins_name='Total', 
    dropna=True, 
    normalize=False)

result.head()

In [ ]:
import pandas as pd

result = pd.crosstab(
    index = data["대여소명"], 
    columns = data["연령대코드"], 
    values= data["이용건수"], 
    aggfunc='mean', 
    margins=True, 
    margins_name='Total', 
    dropna=True, 
    normalize=False)

result.head()

In [ ]:
# summary_data에서 인덱스 리셋하기
summary_data = summary_data.reset_index()
summary_data.head()

In [ ]:
# 반복문 이해하기
lat_list = []  # 위도
long_list = [] # 경도

for name in summary_data["대여소명"].tolist():
    try:
        lat_value = sub_data.loc[sub_data["대여소명"]==name, "위도"].item() # .values[0], .iloc[0]
        long_value = sub_data.loc[sub_data["대여소명"]==name, "경도"].item()
    except:
        lat_value = None
        long_value = None
    
    lat_list.append(lat_value)
    long_list.append(long_value)

summary_data["위도"] = lat_list
summary_data["경도"] = long_list

summary_data    

In [ ]:
# summary_data에서 이동거리가 많은 순으로 Top10 뽑기 data_d_top10
data_d_top10 = summary_data.sort_values(by="이동거리(M)", ascending=False).head(10)
data_d_top10

In [ ]:
# summary_data에서 이용시간이 많은 순으로 Top10 뽑기 data_t_top10
summary_data.sort_values(by="이용시간(분)", ascending=False).head(10)

In [ ]:
# summary_data에서 이용건수가 많은 순으로 Top10 뽑기 data_c_top10
summary_data.sort_values(by="이용건수", ascending=False).head(10)

#### 지도 시각화

In [ ]:
import folium

# 지도 출력하기
center = [37.565516, 126.978025] # 중심 위치
map = folium.Map(location=center, zoom_start=12)
map

#### 이동거리 Top10 표시하기

In [ ]:
for i in range(len(data_d_top10)):
    temp = data_d_top10.iloc[i]
    lat, lng = temp[["위도", "경도"]]

    folium.Marker(location=[lat, lng], icon=folium.Icon(color="red"), popup=temp.name).add_to(map)
map

#### 이용시간 Top10 표시하기

#### 이용건수 Top10 표시하기

#### 머지 데이터로

In [ ]:
new_data = pd.merge(
    left=data,
    right=sub_data,
    on="대여소번호"
)

new_data